[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NadavFireman/Representation-Learning-Midterm-DAE-SimCLR/blob/main/colab_train_MA_A.ipynb)

# Part A: the final runs

The final DAE (salt and pepper 0.5, h 256, dropout 0.4, 20 epochs, the epoch with the best val k-NN is kept) and the AE (the same, `--noise_level 0`), then the full protocol on them, on a random encoder and on the supervised encoder:

- **Supervised baseline**: the same CNN-3 trained with labels, on 1%, 10% and 100% of the labels.
- **k-NN** (k=20, cosine) and **linear probe** of every encoder, on Tiny-ImageNet and CIFAR-100.
- **Fine-tuning** of the DAE, the AE and the random encoder on 1%, 10% and 100% of the labels, on both datasets.
- **Analysis** (A4 robustness and A5 evidence) of every encoder.

Every run writes to `midterm_final` on Drive, a new folder: the earlier final runs in `midterm_runs` have the same names, so they would be skipped. Every run is scored on the test set (only val chooses the checkpoint).

**One seed per runtime**: `SEED` = 42, 1 or 2, in three runtimes at the same time, about 5 to 6 hours each. The mean and standard deviation over the 3 seeds go to the report. After a disconnect, run all again: finished runs are skipped, unfinished ones continue from their `last.ckpt`.

## Setup
Downloads the repository (or pulls the latest code), installs it and checks the GPU. Use an L4 runtime.

In [ ]:
import os

REPO = "/content/Representation-Learning-Midterm-DAE-SimCLR"
if not os.path.exists(REPO):
    !git clone https://github.com/NadavFireman/Representation-Learning-Midterm-DAE-SimCLR.git {REPO}
os.chdir(REPO)
!git pull
!git log --oneline -1

In [ ]:
%pip install -q -r requirements.txt

import torch
assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → L4 GPU, then rerun this cell."
print(torch.cuda.get_device_name(0), "|", os.cpu_count(), "CPU cores")

## Drive
`OUT` is where every run writes. The tuning choices are read from `midterm_runs`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/לימודים/תואר שני/שנה ב/סמסטר א/למידת ייצוגים במודלים עמוקים/מטלות/מטלת אמצע"
OUT = f"{DRIVE_DIR}/midterm_final"
CHOICE_FILE = f"{DRIVE_DIR}/midterm_runs/tuning_choice.json"
os.makedirs(OUT, exist_ok=True)
print("writes to:", OUT)

## TensorBoard
Filter the runs with part of a name (`pretrain`, `linear_probe`, `finetune`, `seed1`), then *Toggle all runs*. The `epoch` section has train, val and test on the same axis; the TEXT tab has each run's `run_info`.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{OUT}"

## The runs
`protocol` lists the train.py commands of one seed; `run_all` runs them one by one. A finished run (it has `results.json`) is skipped. A setting with no tuning choice uses the defaults of `config.py`.

In [ ]:
import json

FRACTIONS = (0.01, 0.1, 1.0)
DATASETS = ("tinyimagenet", "cifar100")
DAE = "--dropout 0.4 --num_epochs 20"  # the final DAE: colab_dae_h.ipynb; the top 3 of colab_dae_top3.ipynb tie within the seed noise
ENCODERS = ["--encoder dae", "--encoder dae --noise_level 0", "--encoder random"]  # DAE, AE, random


def tuned(setting):
    """The flags of the tuning choice for setting, or none (config.py's defaults)."""
    with open(CHOICE_FILE) as f:
        choice = json.load(f).get(setting, {})
    return "".join(f" --{k} {v}" for k, v in choice.items())


def protocol(seed):
    """All the runs of one seed: the DAE and the AE, the supervised encoders, then k-NN, linear probe,
    analysis and fine-tuning."""
    s = f" --seed {seed}"
    sup, probe, ft = tuned("cnn3_supervised"), tuned("cnn3_linear_probe"), tuned("cnn3_finetune")
    runs = [f"--mode pretrain {DAE}", f"--mode pretrain --noise_level 0 {DAE}"]
    runs += [f"--mode pretrain_supervise_encoder --label_fraction {f}{sup}" for f in FRACTIONS]
    for flags in (*ENCODERS, "--encoder supervised"):
        for dataset in DATASETS:
            runs += [f"--mode knn {flags} --dataset {dataset}",
                     f"--mode linear_probe {flags} --dataset {dataset}{probe}",
                     f"--mode analysis {flags} --dataset {dataset}"]
    runs += [f"--mode finetune {flags} --dataset {dataset} --label_fraction {f}{ft}"
             for flags in ENCODERS for dataset in DATASETS for f in FRACTIONS]
    return [flags + s for flags in runs]


def run_all(runs):
    for i, flags in enumerate(runs, 1):
        print(f"\n=== {i}/{len(runs)}: {flags}")
        !python train.py {flags} --output_dir "{OUT}"

## Which seed
42, 1 or 2: one per runtime.

In [ ]:
SEED = 42  # 42, 1 or 2

## Run
47 runs: the DAE and the AE, 3 supervised encoders, 24 evaluations (k-NN, linear probe and analysis of 4 encoders on 2 datasets) and 18 fine-tunings.

In [ ]:
run_all(protocol(SEED))

## Results
Every finished run of `midterm_final`, from its `run_info.txt`. Run it when all three seeds are done.

In [ ]:
import glob

import pandas as pd

rows = []
for path in glob.glob(f"{OUT}/date*/run_info.txt"):
    with open(path, encoding="utf-8") as f:
        rows.append(dict(line.rstrip("\n").split(": ", 1) for line in f if ": " in line))
runs = pd.DataFrame(rows).sort_values("started")
pd.set_option("display.max_rows", None, "display.max_colwidth", 120)
print(len(runs), "runs")
runs[["started", "folder", "epochs", "results"]]